<h1 style="color: red;">Wizualizacja zmiennych</h1>

`seaborn` – biblioteka do wizualizacji zmiennych losowych, bazująca na bibliotece `matplotlib`\
Dzięki integracji z biblioteką `pandas`, czytelne wizualizacje można uzyskać w niej w sposób bardziej zautomatyzowany. \
Nazwa biblioteki powstała od statystyka specjalizującego się w wizualizacji danych - Williama Seaborna Clevelanda.

Instalacja biblioteki:                       

In [ ]:
%pip install seaborn

Załadowanie bibliotek:

In [ ]:
import pandas as pd
import janitor
import numpy as np 
import matplotlib.pyplot as plt

import seaborn as sns
sns.set_theme(style = 'darkgrid')       # ustalenie jednego ze stylów wykresów biblioteki seaborn

---

In [ ]:
# wczytanie danych

data = pd.read_csv('StudentsPerformance.csv')

# czyszczenie danych

data = data.clean_names(
          ).rename(columns = {'race_ethnicity'             :'group',
                              'parental_level_of_education':'parent_education',
                              'test_preparation_course'    :'if_preparation_course'})
data = data.assign(gender                = data['gender'].map({'female':'F', 'male':'M'}),
                   group                 = data['group'].str[-1].apply(lambda x: str(ord(x) - 64)),
                   parent_education      = data['parent_education'].astype('category'
                                                                           ).cat.reorder_categories(['some high school', 'high school',
                                                                                                     'some college', 'associate\'s degree',
                                                                                                     'bachelor\'s degree', 'master\'s degree'],
                                                                                                    ordered = True),
                   if_preparation_course = data['if_preparation_course'].map({'completed':'yes', 'none':'no'}),
                   grade                 = pd.cut(data[['math_score', 'reading_score', 'writing_score']].mean(axis = 'columns'),
                                                  bins = [0, 60, 79, 100], labels = ['C', 'B', 'A']))
data.insert(column = 'group', value = data.pop('group'), loc = data.columns.get_loc('if_preparation_course') + 1)

In [ ]:
# podgląd danych

display(data.head())                    # dane dotyczą wyników uczniów z kilku testów
data.info()

### W pliku:

1. Wykresy rozkładu zmiennej
   - nominalna
   - porządkowa
   - ilościowa
2. Wykresy rozkładu łącznego dwóch zmiennych
   - ilościowa vs ilościowa
   - ilościowa vs porządkowa
   - ilościowa vs nominalna
   - porządkowa vs nominalna
   - nominalna vs nominalna
3. Wykresy rozkładu łącznego więcej niż dwóch zmiennych

---

<h1 style="color: red;">Wykresy rozkładu jednej zmiennej</h1>

**<h2 style="color: darkorange;">Zmienna nominalna</h2>**

Rozkład zmiennej nominalnej można przedstawić na wykresie kołowym lub słupkowym.

<h3 style="color: dodgerblue;">Wykres kołowy</h3>

**<p style="color: darkblue;">Tabela częstości:</p>**

In [ ]:
gender_freq = data['gender'].value_counts() 
gender_freq

**<p style="color: darkblue;">**Wykres surowy:**</p>**

In [ ]:
plt.pie(gender_freq, labels = gender_freq.index)
plt.show()

**<p style="color: darkblue;">Wykres gotowy:</p>**

In [ ]:
plt.figure(figsize = (4.2, 2.6))


plt.pie(gender_freq, labels = ['kobiety', 'mężczyźni'],         # tabela częstości; labels - lista etykiet wartości
        startangle   = 90,                                      # kąt rozpoczęcia wykresu, gdzie 0 to dodatnia część osi x
        counterclock = False,                                   # czy przeciwnie do ruchu wskazówek zegara
        autopct      = '%1.1f%%',                               # dodanie etykiet procentów
        pctdistance  = 0.5,                                     # odległość etykiet procentów od środka, gdzie 1 to obwód
        radius       = 1.2,                                     # promień
        colors       = ['pink', 'lightblue'],                   # kolory pól
        textprops    = {'fontsize': 14})                        # wielkość czcionki tekstu


plt.tight_layout()
plt.savefig('kolowy_plec.png', dpi = 300)
plt.show()

<h3 style="color: dodgerblue;">Wykres słupkowy</h3>

**<p style="color: darkblue;">**Wykres surowy:**</p>**

In [ ]:
sns.countplot(data, x = 'group')
plt.show()

**<p style="color: darkblue;">Wykres gotowy:</p>**

In [ ]:
sns.countplot(data, x = 'group',
              order = sorted(data['group'].unique()),           # lista wartości definiująca porządek słupków, tu porządek alfabetyczny
              color = 'darkorange')

plt.xlabel('\ngrupa',    size = 16)
plt.ylabel('częstość\n', size = 16)

plt.xticks(size = 16)
plt.yticks(size = 16)

plt.tight_layout()
plt.savefig('slupkowy_grupa.png', dpi = 300)
plt.show()

**<h2 style="color: darkorange;">Zmienna porządkowa</h2>**

Rozkład zmiennej porządkowej również można przedstawić na wykresie słupkowym, ale z etykietami w kolejności poziomów tej zmiennej.

<h3 style="color: dodgerblue;">Wykres słupkowy</h3>

**<p style="color: darkblue;">**Wykres surowy:**</p>**

In [ ]:
sns.countplot(data, x = 'parent_education')
plt.show()

Problem: ?

Rowiązanie 1:

In [ ]:
plt.figure(figsize = (11, 4))

sns.countplot(data, x = 'parent_education')
plt.show()

Rowiązanie 2:

In [ ]:
sns.countplot(data, x = 'parent_education')
plt.xticks(rotation = 45, ha = 'right')                         # obrót etykiet na osi x o 45 stopni, tak by prawy koniec był pod słupkiem
plt.show()

Rowiązanie 3:

In [ ]:
sns.countplot(data, y = 'parent_education')                     # poziomy zmiennej na osi y
plt.show()

Rozwiązanie 4: 

In [ ]:
data['parent_education'].cat.categories

In [ ]:
education_levels = data['parent_education'].cat.categories.str.replace(' ', '\n')
education_levels

**<p style="color: darkblue;">Wykres gotowy:</p>**

In [ ]:
plt.figure(figsize = (7, 5))

sns.countplot(data, x = 'parent_education',
              color = 'olive')

plt.xlabel('\nwykształcenie rodziców [poziom]', size = 14)
plt.ylabel('częstość\n',                        size = 14)

plt.xticks(range(6), education_levels, size = 11)               # drugi argument - lista podpisów punktów na osi x
plt.yticks(size = 13)

plt.tight_layout()
plt.savefig('slupkowy_wyksztalcenie.png', dpi = 300)
plt.show()

**<h2 style="color: darkorange;">Zmienna ilościowa</h2>**

Rozkład zmiennej ilościowej można przedstawić na histogramie lub wykresie pudełkowym.

<span style="color:red">Uwaga:</span> Jeśli zmienna jest zmienną ilościową dyskretną o niewielkiej liczbie poziomów, adekwatnymi wykresami jej rozkładu są wykresy dla zmiennej porządkowej.\
Tyczy się to również rozkładów łącznych takiej zmiennej z inną zmienną.

<h3 style="color: dodgerblue;">Histogram</h3>

Histogram podobnie jak wykres słupkowy przedstawia w postaci słupków częstość poszczególnych wartości, jednak nie dla każdej z osobna, a należących wspólnie do przedziału.

Histogram może być **<span style="color: darkblue;">częstościowy</span>** lub **<span style="color: darkblue;">gęstościowy</span>** w zależności od tego, co przedstawia jego oś y.

Oś y histogramu częstościowego, analogicznie do wykresu słupkowego, przedstawia liczbę obserwacji znajdujących się w danym przedziale.\
Oś y histogramu gęstościowego jest natomiast przeskalowana tak, by pole wszystkich słupków wynosiło 1\
(oś y przedstawia odsetek obserwacji w danym przedziale wśród wszystkich obserwacji podzielony przez długość przedziału).

Pozostałe elementy obu typów histogramów są takie same - histogram gęstościowy jest tylko znormalizowaną wersją histogramu częstościowego.

**<p style="color: darkblue;">**Wykres surowy:**</p>**

In [ ]:
sns.histplot(data, x = 'math_score')
plt.show()

**<p style="color: darkblue;">Wykres gotowy:</p>**

In [ ]:
sns.histplot(data, x = 'math_score',
             stat  = 'density',                                 # typ histogramu, tu gęstościowy
             bins  = np.arange(0, 101, 5),                      # lista punktów wyznaczających granice przedziałów
             color = 'gold',
             alpha = 1)                                         # nieprzezroczystość z przedziału [0, 1], tu pełna (1) dla nieprześwitywania siatki

plt.xlabel('\nwynik testu z matematyki [pkt]', size = 16)
plt.ylabel('gęstość\n',                        size = 16)

plt.xticks(size = 14)
plt.yticks(size = 14)

plt.ylim(0, 0.0315)

plt.tight_layout()
plt.savefig('histogram_matematyka.png', dpi = 300)
plt.show()

<h3 style="color: dodgerblue;">Wykres pudełkowy</h3>

Wykres pudełkowy jest wykresem jednowymiarowym, przedstawiającym na osi kwantyle rozkładu zmiennej ilościowej oraz jego wartości odstające.

Linie tworzące granice pudełka to kwartyle $Q_1$, $Q_2$ (mediana), $Q_3$.

Wąsy wykresu odchodzą od $Q_1$ i $Q_3$ o 1,5 wartości rozstępu międzykwartylowego lub do odpowiednio najmniejszej lub największej wartości zaobserwowanej w danych.

Obserwacje, które znajdą się poza wąsami wykresu - obserwacje odstające zmiennej, oznacza się kropkami.

          Q1 - 1.5 IQR        Q1  mediana  Q3         Q3 + 1.5 IQR
              
                               |-----|-----| 
         o     |---------------|     |     |---------------|     o  o
                               |-----|-----| 
                           
      wartość                  <----------->                   wartości
     odstająca                      IQR                       odstające

**<p style="color: darkblue;">Kwantyle rozkładu zmiennej:</p>**

In [ ]:
data['math_score'].quantile([0, 0.25, 0.5, 0.75, 1])

**<p style="color: darkblue;">**Wykres surowy:**</p>**

In [ ]:
sns.boxplot(data, x = 'math_score')
plt.show()

**<p style="color: darkblue;">Wykres gotowy:</p>**

In [ ]:
plt.figure(figsize = (6, 2.75))

sns.boxplot(data, x = 'math_score',                             # jeśli x, wykres poziomy; jeśli y, wykres pionowy
            width = 0.4,                                        # szerokość pudełka
            color = 'yellow')

plt.xlabel('\nwynik testu z matematyki [pkt]', size = 14)

plt.xticks(size = 14)                                           # brak yticks, bo brak punktów na drugiej osi

plt.tight_layout()
plt.savefig('pudelkowy_matematyka.png', dpi = 300)
plt.show()

------

*<h3 style="color: forestgreen;"> Ćwiczenia ✏️ </h3>*

1. Stwórz wykres kołowy rozkładu zmiennej `if_preparation_course`.

2. Stwórz wykres rozkładu zmiennej `grade`.

3. Stwórz wykres rozkładu zmiennej `reading_score` typu:
* histogram

* wykres pudełkowy

---

<h1 style="color: red;">Wykresy rozkładu łącznego dwóch zmiennych</h1>

**<h2 style="color: darkorange;">Ilościowa vs Ilościowa</h2>**

Rozkład dwóch zmiennych ilościowych względem siebie można przedstawić na przykład za pomocą wykresu rozrzutu.

<h3 style="color: dodgerblue;">Wykres rozrzutu</h3>

Jeśli mamy zmienne $x$ i $y$, których wartości w kolumnie to odpowiednio: [$x_1$, $x_2$, ... , $x_n$] i [$y_1$, $y_2$, ... , $y_n$], to punkty $(x_1, y_1)$, $(x_2, y_2)$, ... , $(x_n, y_n)$ na układzie współrzędnych, tworzą ich wykres rozrzutu.

**<p style="color: darkblue;">**Wykres surowy:**</p>**

In [ ]:
sns.scatterplot(data, x = 'math_score', y = 'reading_score')
plt.show()

**<p style="color: darkblue;">Wykres gotowy:</p>**

In [ ]:
plt.style.use('default')                                    # zmiana stylu wykresów dla widoczności przejrzystości

plt.figure(figsize = (5.5, 5.5)) 
plt.grid(True)


sns.scatterplot(data, x = 'math_score', y = 'reading_score',
                color     = 'green',
                edgecolor = 'black',                        # kolor obramowania punktów
                alpha     = 0.27,                           # nieprzezroczystość z przedziału [0, 1], tu niska dla widoczności pokrywających się punktów
                s         = 8)                              # rozmiar punktów


plt.xlabel('\nwynik testu z matematyki [pkt]', size = 15)
plt.ylabel('wynik testu z czytania [pkt]\n',   size = 15)

plt.xticks(size = 13)
plt.yticks(size = 13)

plt.ylim(-5, 105)

plt.tight_layout()
plt.savefig('rozrzutu_matematyka_czytanie.png', dpi = 300)
plt.show()

sns.set_theme(style = 'darkgrid')                           # powrót do wcześniejszego stylu wykresów 

**<h2 style="color: darkorange;">Porządkowa vs Porządkowa</h2>**

Rozkład dwóch zmiennych porządkowych względem siebie można przedstawić na przykład za pomocą mapy ciepła.

<h3 style="color: dodgerblue;">Mapa ciepła</h3>

Mapa ciepła, analogicznie do wykresu słupkowego, pokazuje częstość danej pary wartości tych zmiennych, jednak przedstawia to za pomocą nie wysokości słupka dla poszczególnych wartości na osi, a określonego odcienia koloru na przestrzeni osi obu zmiennych.

**<p style="color: darkblue;">**Tabela krzyżowa:**</p>**

In [ ]:
reading_education_freq = pd.crosstab(data['grade'], data['parent_education'])
reading_education_freq

**<p style="color: darkblue;">**Wykres surowy:**</p>**

In [ ]:
sns.heatmap(reading_education_freq)
plt.show()

**<p style="color: darkblue;">Wykres gotowy:</p>**

In [ ]:
plt.figure(figsize = (7, 5))

sns.heatmap(reading_education_freq.iloc[::-1],                             # tabela krzyżowa, tu z odwróceniem kolejności wierszy
            vmin = 0, vmax = 120,                                          # przedział wartości częstości, na których rozpięta ma być skala odcieni
            cbar_kws = {'orientation':'horizontal',                        # ustawienia paska kolorów: orientacja, umiejscowienie, podpis 
                        'location'   :'top',
                        'label'      :'częstość'},
            cmap     = 'viridis')                                          # paleta kolorów mapy

plt.xlabel('\nwykształcenie rodziców [poziom]', size = 14)
plt.ylabel('ocena\n',    size = 14)

plt.xticks(np.arange(0.5, 6), education_levels, rotation = 0, size = 11)   # rotation - kąt nachylenia etykiet poziomów, tu poprawa do poziomego 
plt.yticks(rotation = 0, size = 11)

plt.tight_layout()
plt.savefig('ciepla_wyksztalcenie_czytanie.png', dpi = 300)
plt.show()

**<h2 style="color: darkorange;">Ilościowa  vs Porządkowa</h2>**

Rozkład zmiennej ilościowej względem zmiennej porządkowej można przedstawić agregując zmienną ilościową dla poziomów zmiennej porządkowej, a następnie przedstawiając to na wykresie słupkowym. Innym rozwiązaniem jest stworzenie tzw. wykresu panelowego, tj. panelu wykresów rozkładu zmiennej ilościowej (histogramów lub wykresów pudełkowych), gdzie kolejne z nich odpowiadają kolejnym poziomom zmiennej porządkowej.

<h3 style="color: dodgerblue;">Wykres słupkowy (z agregacją)</h3>

Wysokość słupków na wykresie słupkowym prezentuje zazwyczaj wartość statystyki będącej miarą tendencji centralnej, natomiast opcjonalne słupki błędów mogą przedstawiać, ile wyniosłaby ta statystyka, gdyby odjąć lub dodać do niej statystykę będącą jakąś miarą rozrzutu. Zazwyczaj tą drugą jest błąd standardowy, odchylenie standardowe lub połowa długości przedziału ufności. Istotne jest, aby zamieszczając gdzieś wykres słupkowy ze słupkami błędów, poinformować w podpisie wykresu o tym, co one przedstawiają.

**<p style="color: darkblue;">**Wykres surowy:**</p>**

In [ ]:
sns.barplot(data, x = 'parent_education', y = 'writing_score')
plt.show()

**<p style="color: darkblue;">Wykres gotowy:</p>**

In [ ]:
plt.figure(figsize = (7, 4.7))

sns.barplot(data, x = 'parent_education', y = 'writing_score',             # y - zmienna agregowana
            estimator = 'mean',                                            # funkcja agregująca
            errorbar  = 'se',                                              # słupki błędów, tu błędy standardowe 
            err_kws   = {'linewidth': 1.5},                                # ustawienia słupków błędów
            color     = 'yellowgreen')

plt.xlabel('\nwykształcenie rodziców [poziom]',      size = 14)
plt.ylabel('średni wynik z testu z pisania [pkt]\n', size = 14)

plt.xticks(range(6), education_levels, size = 12)
plt.yticks(size = 13)

plt.tight_layout()
plt.savefig('slupkowy_wyksztalcenie_pisanie.png', dpi = 300)
plt.show()

<h3 style="color: dodgerblue;">Panel wykresów pudełkowych</h3>

**<p style="color: darkblue;">**Wykres surowy:**</p>**

In [ ]:
sns.boxplot(data, x = 'parent_education', y = 'writing_score')
plt.show()

**<p style="color: darkblue;">Wykres gotowy:</p>**

In [ ]:
plt.figure(figsize = (7, 5.5))

sns.boxplot(data, x = 'parent_education', y = 'writing_score',             # jeśli ilościowa na x, wykres poziomuy; jeśli ilościowa na y, wykres pionowy
            width = 0.6,
            color = 'yellow')

plt.xlabel('\nwykształcenie rodziców [poziom]', size = 14)
plt.ylabel('wynik testu z pisania [pkt]\n',     size = 14)

plt.xticks(range(6), education_levels, size = 11)
plt.yticks(size = 13)

plt.ylim(-10, 110)

plt.tight_layout()
plt.savefig('pudelkowy_wyksztalcenie_pisanie.png', dpi = 300)
plt.show()

<h3 style="color: dodgerblue;">Panel histogramów</h3>

Histogramy znajdujące się na jednym wykresie panelowym powinny być typu gęstościowego, aby zrównoważyć nierówność w liczności poszczególnych grup.

**<p style="color: darkblue;">Wykres surowy:</p>**

In [ ]:
sns.displot(data, x = 'reading_score', col = 'parent_education',
            kind  = 'hist')
plt.show()

**<p style="color: darkblue;">Wykres gotowy:</p>**

In [ ]:

                                                              # zapis wykresu do zmiennej dla manipulacji właściwościami poszczególnych wykresów
hists = sns.displot(data, x = 'reading_score', col = 'parent_education',         # col - zmienna grupująca
                    col_wrap = 3, height = 3, aspect = 1.1,   # col_wrap - l. kolumn w panelu, height - wysok. 1 wykresu; aspect = stosunek szer./wys.
                    kind  = 'hist',                           # typ wykresu na panelu, tu histogram
                    stat  = 'density', common_norm = False,   # ustawienia histogramu: typ, tu gęstościowy bez wspólnej normalizacji dla porównywalności
                    bins  = np.arange(0, 101, 5),             #                        lista punktów wyznaczających granice przedziałów
                    color = 'cornflowerblue',                 #
                    alpha = 1)                                #                        nieprzezroczystość dla nieprześwitywania siatki

hists.set_axis_labels('\nwynik testu z czytania [pkt]', 'gęstość\n', size = 12)  # ustawienia wykresów; odpowiednik plt.xlabel i plt.ylabel
hists.set(ylim   = (0, 0.04),                                                    # odpowiednik plt.xlim i plt.ylim
          xticks = np.arange(0, 101, 20))                                        # odpowiednik plt.xticks i plt.yticks
for ax in hists.axes.flat:                                                       # pozostałe ustawienia xticks i yticks
    ax.tick_params(axis = 'both', labelsize = 10, labelbottom = True)            # labelbottom - czy punkty na osi x mają być na każdym wykresie
hists.set_titles('\n{col_name}\n', size = 14)          # odpowiednik plt.title; tu {col_name} - automatyczne nazwanie wartością zmiennej z argumentu col

plt.suptitle('Wykształcenie rodziców [poziom]', size = 15)                       # podpis panelu

plt.tight_layout()
plt.savefig('panel_czytanie_wyksztalcenie.png', dpi = 300)
plt.show()

------

*<h3 style="color: forestgreen;"> Ćwiczenia ✏️ </h3>*

1. Stwórz wykres rozkładu łącznego zmiennych `writing_score` i `reading_score`.

2. Stwórz mapę ciepła rozkładu łącznego zmiennych `grade` i `group`.

3. Stwórz wykres rozkładu łącznego zmiennych `math_score` i `grade`typu:
- wykres słupkowy z agregacją

- panel wykresów pudełkowych 

- panel histogramów

---

**<h2 style="color: darkorange;">Ilościowa vs Nominalna</h2>**

Aby przedstawić rozkład zmiennej ilościowej względem zmiennej nominalnej możemy skorzystać z podobnego zbioru wykresów, co dla rozkładu tej zmiennej względem zmiennej porządkowej - wykonać agregację zmiennej dla każdego poziomu zmiennej nominalnej lub skorzystać z panelu wykresów dla poszczególnych z nich. Przy mniejszej liczbie poziomów zmiennej nominalnej, panel histogramów można zastąpić przedstawieniem kilku histogramów na jednym wykresie.

<h3 style="color: dodgerblue;">Wykres słupkowy</h3>

In [ ]:
plt.figure(figsize = (4.5, 5))

sns.barplot(data, x = 'if_preparation_course', y = 'writing_score',
            order    = ['yes', 'no'],                                      # lista wartości definiująca porządek wykresów
            errorbar = 'se',
            err_kws  = {'linewidth': 1.5},
            color    = 'yellowgreen')

plt.xlabel('\nczy kurs przygotowawczy do testów',    size = 13)
plt.ylabel('średni wynik z testu z pisania [pkt]\n', size = 14)

plt.xticks(range(2), ['Tak', 'Nie'], size = 14)
plt.yticks(size = 14)

plt.xlim(-1, 2)

plt.tight_layout()
plt.savefig('slupkowy_kurs_pisanie.png', dpi = 300)
plt.show()

<h3 style="color: dodgerblue;">Panel wykresów pudełkowych</h3>

In [ ]:
plt.figure(figsize = (6, 3.5))

sns.boxplot(data, x = 'writing_score', y = 'if_preparation_course',
            order = np.sort(data['if_preparation_course'].unique()),       # lista wartości definiująca porządek słupków, tu porządek alfabetyczny
            width = 0.35,
            color = 'yellow') 

plt.xlabel('\nwynik testu z pisania [pkt]',       size = 14)
plt.ylabel('czy kurs przygotowawczy do testów\n', size = 14)

plt.xticks(size = 14)
plt.yticks(range(2), ['Nie', 'Tak'], size = 14)

plt.xlim(-10, 110)

plt.tight_layout()
plt.savefig('pudelkowy_pisanie_kurs.png', dpi = 300)
plt.show()

<h3 style="color: dodgerblue;">Panel histogramów</h3>

In [ ]:
hists = sns.displot(data, x = 'math_score', col = 'group',
                    col_order = np.sort(data['group'].unique()),           # lista wartości definiująca porządek słupków - porządek alfabetyczny
                    col_wrap = 3, height = 3, aspect = 1.1, 
                    kind  = 'hist',
                    stat  = 'density', common_norm = False,
                    bins  = np.arange(0, 101, 5),
                    color = 'cornflowerblue',
                    alpha = 1)

hists.set_axis_labels('\nwynik testu z matematyki [pkt]', 'gęstość\n', size = 12)
hists.set(xticks = np.arange(0, 101, 20),
          ylim   = (0, 0.034)) 
for ax in hists.axes.flat:
    ax.tick_params(axis = 'both', labelsize = 10, labelbottom = True)
hists.set_titles('\n{col_name}\n', size = 14)

plt.suptitle('Grupa', size = 15) 

plt.tight_layout()
plt.savefig('panel_matematyka_grupa.png', dpi = 300)
plt.show()

<h3 style="color: dodgerblue;">Histogramy nakładające się</h3>

Histogramy nakładające się na jednym wykresie powinny być typu gęstościowego, aby zrównoważyć nierówność w liczności poszczególnych grup.

**<p style="color: darkblue;">**Wykres surowy:**</p>**

In [ ]:
sns.histplot(data, x = 'writing_score', hue = 'if_preparation_course')
plt.show()  

**<p style="color: darkblue;">Wykres gotowy:</p>**

In [ ]:
plt.grid(False)                                                            # usunięcie siatki dla nieprześwitywania jej pod wykresami

sns.histplot(data, x = 'writing_score', hue = 'if_preparation_course',     # hue - zmienna grupująca
             stat    = 'density', common_norm = False,                     # typ histogramu, tu gęstościowy bez wspólnej normalizacji dla porównywalności
             bins    = np.arange(0, 101, 5),
             palette = {'yes':'orangered',                                 # kolory wykresów poszczególnych grup
                        'no' :'dodgerblue'})

plt.legend(title          = 'Czy kurs przygotowawczy',
           labels         = ['Tak', 'Nie'],
           title_fontsize = 11.5,
           fontsize       = 11.5,
           facecolor      = 'white',
           framealpha     = 0.9)

plt.xlabel('\nwynik testu z pisania [pkt]', size = 16)
plt.ylabel('gęstość\n',                     size = 16)

plt.xticks(size = 14)
plt.yticks(size = 14)

plt.ylim(0, 0.037)

plt.tight_layout()
plt.savefig('histogram_pisanie_kurs.png', dpi = 300)
plt.show()

**<h2 style="color: darkorange;">Porządkowa vs Nominalna</h2>**

Gdy chcemy zwizualizować rozkład zmiennej porządkowej względem zmiennej nominalnej, tak jak przy zmiennej ilościowej, możemy przedstawić to przy pomocy kilku rozkładów na jednym wykresie lub skorzystać z panelu wykresów (ale tym razem słupkowych), a także tak jak przy dwóch zmiennych porządkowych, jeśli zmienna nominalna ma wiele poziomów, skorzystać ewentualnie z mapy ciepła.

<h3 style="color: dodgerblue;">Wykres słupkowy grupowany</h3>

In [ ]:
plt.figure(figsize = (7.3, 4.5))

sns.countplot(data, x = 'parent_education', hue = 'lunch',                 # hue - zmienna grupująca
              palette = {'standard'    :'orange',                          # kolory wykresów poszczególnych grup
                         'free/reduced':'dodgerblue'}) 

plt.legend(title          = 'Lunch',
           labels         = ['standardowy', 'darmowy/tańszy'],
           title_fontsize = 11.5,
           fontsize       = 11,
           facecolor      = 'white',
           framealpha     = 1)

plt.xlabel('\nwykształcenie rodziców [poziom]', size = 14)
plt.ylabel('częstość\n',                        size = 14)

plt.xticks(range(6), education_levels, size = 11)
plt.yticks(size = 13)

plt.xlim(-0.65, 5.65)

plt.tight_layout()
plt.savefig('slupkowy_grp_wyksztalcenie_lunch.png', dpi = 300)
plt.show()

<h3 style="color: dodgerblue;">Wykres słupkowy skumulowany</h3>

**<p style="color: darkblue;">**Wykres surowy:**</p>**

In [ ]:
sns.histplot(data = data, x = 'parent_education', hue = 'lunch',
             multiple = 'stack')
plt.show()

**<p style="color: darkblue;">Wykres gotowy:</p>**

In [ ]:
plt.figure(figsize = (7, 4.5))
plt.grid(axis = 'x')

sns.histplot(data = data, x = 'parent_education', hue = 'lunch',     # hue - zmienna grupująca
             multiple = 'stack',                                     # sposób ustawienia rozkładu grup, tu skumulowany
             shrink   = 0.8,                                         # szerokość słupka z przedziału [0, 1], tu niższa niż 1 dla odstępów między słupkami
             palette  = {'standard'    :'orange',                    # kolory wykresów poszczególnych grup
                         'free/reduced':'dodgerblue'},
             alpha    = 1)                                           # nieprzezroczystość z przedziału [0, 1], tu pełna (1) dla nieprześwitywania siatki

plt.legend(title          = 'Lunch',
           labels         = ['darmowy/tańszy', 'standardowy'],
           title_fontsize = 11.5,
           fontsize       = 11,
           facecolor      = 'white',
           framealpha     = 1)

plt.xlabel('\nwykształcenie rodziców [poziom]', size = 14)
plt.ylabel('częstość\n',                        size = 14)

plt.xticks(range(6), education_levels, size = 11)
plt.yticks(size = 13)

plt.tight_layout()
plt.savefig('slupkowy_wyksztalcenie_lunch.png', dpi = 300)
plt.show()

<h3 style="color: dodgerblue;">Panel wykresów słupkowych</h3>

Jeśli poziomy zmiennej porządkowej mają długie nazwy, dla ich czytelności można stworzyć odpowiednio wysoki panel lub zamieścić w etykietach słupków jedynie ich kolejne rangi (w poniższym przykładzie wynoszą one od 1 do 6), a ich pełne opisy załączyć w podpisie wykresu panelowego.

**<p style="color: darkblue;">Wykres surowy:</p>**

In [ ]:
sns.catplot(data, x = 'parent_education', col = 'lunch',
            kind = 'count')
plt.show()

**<p style="color: darkblue;">Wykres gotowy:</p>**

In [ ]:
data_mapped = data.assign(lunch = data['lunch'].map({'standard'    :'standardowy',       # ramka ze zmienionymi poziomami zmiennej
                                                     'free/reduced':'darmowy/tańszy'}))  #   dla wygody zmiany etykiet na j. polski
                                                                 # zapis wykresu do zmiennej dla manipulacji właściwościami poszczególnych wykresów 
bars = sns.catplot(data_mapped, x = 'parent_education', col = 'lunch',                   #  col - zmienna grupująca  
                   col_wrap = 1, height = 3.65, aspect = 1.5,    # col_wrap - l. kolumn w panelu, height - wysok. 1 wykresu; aspect = stosunek szer./wys.
                   kind  = 'count',                                                      # typ wykresu na panelu, tu słupkowy częstościowy
                   color = 'olive')

bars.set_axis_labels('\nwykształcenie rodziców [poziom]', 'częstość\n', size = 11.5)     # ustawienia wykresów; odpowiednik plt.xlabel i plt.ylabel
bars.set(ylim = (0, 160))                                                                # odpowiednik plt.xlim i plt.ylim
for ax in bars.axes.flat:                                                                # odpowiednik plt.xticks i plt.yticks
    ax.tick_params(axis = 'x', labelsize = 9.7, labelbottom = True)                      # labelbottom - czy punkty na osi x mają być na każdym wykresie 
    ax.tick_params(axis = 'y', labelsize = 10)
    ax.set_xticks(range(6))
    ax.set_xticklabels(education_levels)
bars.set_titles('\n{col_name}\n', size = 13)                                      # odpowiednik plt.title
                                                                                  # {col_name} - automatyczne nazwanie wartością zmiennej z argumentu col
plt.suptitle('Lunch', size = 14)                                                  # podpis panelu

plt.tight_layout()
plt.savefig('panel_wyksztalcenie_lunch.png', dpi = 300)
plt.show()

**<h2 style="color: darkorange;">Nominalna vs Nominalna</h2>**

Rozkład dwóch zmiennych nominalnych względem siebie można przedstawić podobnie jak rozkład zmiennej porządkowej względem innej zmiennej porządkowej lub zmiennej nominalnej - za pomocą mapy ciepła lub wykresów słupkowych takich jak wyżej (skumulowanych lub grupowanych). Inną opcją, z której można skorzystać, jest na przykład panel wykresów kołowych.

Jeśli obie zmienne mają wiele poziomów, to spośród tych możliwości najbardziej preferowaną opcją ze względu na czytelność, będzie często mapa ciepła, natomiast najmniej preferowaną - wykres słupkowy skumulowany lub grupowany. W pozostałych przypadkach wybór będzie prawdopodobnie zależał od konkretnej charakterystyki danych.

<h3 style="color: dodgerblue;">Panel wykresów kołowych</h3>

W przypadku panelu wykresów kołowych może być dobrze zrobić wspólną legendę, aby nie powtarzać wielokrotnie tych samych etykiet.

In [ ]:
data_mapped = data.assign(gender = data['gender'].map({'F':'kobiety',          # ramka ze zmienionymi poziomami zmiennej
                                                       'M':'mężczyźni'}))

var_summed         = 'gender'                                                  # nazwa zmiennej nominalnej będącej na wykresie kołowym
var_grouping       = 'group'                                                   # nazwa zmiennej nominalnej grupującej

var_summed_label   = 'Płeć'                                                    # podpis zmiennej nominalnej będącej na wykresie kołowym (podpis legendy)
var_grouping_label = 'Grupa'                                                   # podpis zmiennej nominalnej grupującej (tytuł panelu)

In [ ]:
plt.figure(figsize = (7.4, 6.3))

for i, level in enumerate(np.sort(data_mapped[var_grouping].unique())):
    plt.subplot(2, 3, i + 1)

    plt.title('\n' + level + '\n', size = 16)
    
    plt.pie(data_mapped.loc[data_mapped[var_grouping] == level, var_summed].value_counts(),
            startangle   = 90,
            counterclock = False,
            autopct      = '%1.1f%%',
            pctdistance  = 0.5,
            radius       = 1.1,
            colors       = ['pink', 'lightblue'],
            wedgeprops   = {'edgecolor':'white'},
            textprops    = {'size':14})
    plt.tight_layout()

plt.gcf().legend(np.sort(data_mapped[var_summed].unique()),                    # legenda do wszystkich wykresów; gcf "wychodzi" z ostatniego subplot
                 title          = var_summed_label + '\n',
                 title_fontsize = 17,
                 fontsize       = 15,
                 frameon        = False,
                 loc            = 'lower right')                               # umiejscowienie legendy, tu w miejscu pustego pola w siatce

plt.suptitle(var_grouping_label, size = 18)


plt.tight_layout()
plt.savefig('panel_plec_grupa.png', dpi = 300)
plt.show()

W zależności od tego na czym na zależy możemy przy przedstawieniu takiego rozkładu, jako zmienną grupującą wybrać jedną lub drugą zmienną nominalną.

Poniżej znajduje się panel z tymi samymi zmiennymi, ale zamienionymi miejscami. Nie ma w nim pustego pola, więc legenda jest dodana z boku wykresu.

In [ ]:
data_mapped = data.assign(gender = data['gender'].map({'F':'kobiety',
                                                       'M':'mężczyźni'}))

var_summed         = 'group'
var_grouping       = 'gender'

var_summed_label   = 'Grupa'
var_grouping_label = 'Płeć'

In [ ]:
plt.figure(figsize = (6.3, 3.5))

for i, level in enumerate(np.sort(data_mapped[var_grouping].unique())):
    plt.subplot(1, 2, i + 1)

    plt.title('\n' + level + '\n', size = 14)
    
    plt.pie(data_mapped.loc[data_mapped[var_grouping] == level, var_summed].value_counts(),
            startangle   = 90,
            counterclock = False,
            autopct      = '%1.1f%%',
            pctdistance  = 1.4,
            radius       = 0.75,
            colors       = ['#e6c619', '#98df8a', '#34c3e9', '#aec7e8', '#ffc0cb'],
            wedgeprops   = {'edgecolor':'white'},
            textprops    = {'size':11.5})


plt.gcf().legend(np.sort(data_mapped[var_summed].unique()),
                 title          = var_summed_label + '\n',
                 title_fontsize = 14,
                 fontsize       = 12,
                 frameon        = False,
                 loc            = 'right',
                 bbox_to_anchor = (1.15, 0.5))                          # umiejscowienie legendy, gdzie (0, 0) to lewy dolny róg, a (1, 1) prawy górny,
                                                                        #   tu w punkcie (1.15, 0.5) (o 0.15 na prawo od wykresu i w połowie wysokości)
plt.suptitle(var_grouping_label, size = 16)


plt.tight_layout()
plt.savefig('panel_grupa_plec.png', dpi = 300, bbox_inches = 'tight')   # zapis wykresu razem z legendą
plt.show()

------

*<h3 style="color: forestgreen;"> Ćwiczenia ✏️ </h3>*

1. Stwórz panel wykresów pudełkowych rozkładu łącznego zmiennych `reading_score` i `group`.

2. Stwórz nakładające się histogramy dla rozkładu łącznego zmiennych `math_score` i `lunch`.

3. Stwórz wykres rozkładu łącznego zmiennych `grade` i `if_preparation_course` typu:
- wykres słupkowy grupowany

- wykres słupkowy skumulowany

- panel wykresów słupkowych

---

<h1 style="color: red;">Wykresy rozkładu łącznego więcej niż dwóch zmiennych</h1>

Aby uwzględnić na wykresie rozkładu kolejną zmienną można skorzystać z następujących, już wspomnianych wcześniej sposobów:
- zamieścić kilka rozkładów na jednym wykresie, dla każdego poziomu jednej ze zmiennych,
- zamieścić kilka wykresów na jednym panelu, dla każdego poziomu jednej ze zmiennych,
- dodanie dodatkowego wymiaru do wykresu dla jednej ze zmiennych.

W szczególności, w przypadku wykresu rozrzutu, możemy uwzględnić do dwóch dodatkowych zmiennych, poprzez zamianę arbitralnego koloru, kształtu i rozmiaru punktów na elastyczne w zależności od wartości podanych zmiennych - w przypadku dwóch pierwszych cech, zmiennych nominalnych, a w przypadku ostatniej cechy, zmiennej ilościowej:

In [ ]:
plt.style.use('default')

plt.figure(figsize = (8.5, 5.5))
plt.axes().set_aspect('equal')                                               # ustawienie tej samej skali na obu osiach, aby zachować proporcje
plt.grid()                                                                   #  (potrzebne, gdy zbiory wartości zmiennych są pokrewne)

sns.scatterplot(data = data, x = 'math_score', y = 'reading_score',
                hue  = 'lunch', style = 'gender', size = 'writing_score',    # kolor (hue) i kształt (style) zależny od nomin., a rozmiar (size) od iloś.
                sizes     = (10, 50),                                        # przedział wielkości punktów
                palette   = {'free/reduced': 'blue', 'standard': 'red'},     # kolory wykresów poszczególnych grup
                edgecolor = 'white',
                alpha     = 0.3)

legend = plt.legend(title          = 'Legenda\n',                            # zapis do zmiennej dla manipulacji etykietami wartości
                    title_fontsize = 13,
                    fontsize       = 11,
                    facecolor      = '#FDFDFD',
                    framealpha     = 1,
                    bbox_to_anchor = (1.13, 1.015))
for text_old, text_new in zip(legend.get_texts(),                            # zmiana etykiet na polskie (obecne są w legend.get_texts())
                              ['Lunch\n', 'standardowy', 'darmowy/tańszy',
                               '\nWynik z czytania [pkt]\n', '15', '30', '45', '60', '75', '90',
                               '\nPłeć\n', 'kobieta', 'mężczyzna']):
    text_old.set_text(text_new)

plt.xlabel('\nwynik testu z matematyki [pkt]', size = 15)
plt.ylabel('wynik testu z czytania [pkt]\n',   size = 15)

plt.ylim(-5, 105)

plt.tight_layout()
plt.savefig('rozrzutu_matematyka_czytanie_pisanie_lunch.png', dpi = 300, bbox_inches = 'tight')
plt.show()

sns.set_theme(style = 'darkgrid')

---

**<p style="color: red;">Miłego popołudnia!</p>**